# YOLOv8

Install YOLOv8



In [1]:
!pip install -q ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 56.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.7/77.7 kB 5.0 MB/s eta 0:00:00


Check the installation:

In [2]:
import ultralytics

print("Ultralytics version:", ultralytics.__version__)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics version: 8.4.153


Import Libraries

In [3]:
import os
import random
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from collections import Counter

from ultralytics import YOLO

Check GPU Availability

In [4]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Training will run on CPU.")

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


Load YOLOv8

In [5]:
model = YOLO("yolov8n.pt")

print("YOLOv8 model loaded successfully.")

YOLOv8 model loaded successfully.


In [6]:
model.info()

YOLOv8n summary: 129 layers, 3,157,200 parameters, 0 gradients, 8.9 GFLOPs


(129, 3157200, 0, 8.8550912)

In [8]:
import os
from pathlib import Path

# In Kaggle, datasets are automatically extracted to /kaggle/input/
input_dir = Path("/kaggle/input")
train_img_path = None
val_img_path = None

# Dynamically search for the 'train' and 'valid' image directories
for path in input_dir.rglob("*"):
    if path.is_dir() and path.name == "train":
        train_img_path = path.as_posix()
    elif path.is_dir() and path.name == "valid":
        val_img_path = path.as_posix()

if not train_img_path:
    print("Error: Could not find the 'train' folder. Make sure the dataset is added to your Kaggle notebook.")
else:
    # If the folders contain 'images' subfolders, append it
    if os.path.exists(os.path.join(train_img_path, "images")):
        train_img_path = os.path.join(train_img_path, "images")
        val_img_path = os.path.join(val_img_path, "images")

    # Create the data.yaml file in the writable working directory
    yaml_content = f"""
train: {train_img_path}
val: {val_img_path}

nc: 5
names: ['helmet', 'no_helmet', 'no_vest', 'person', 'vest']
"""

    yaml_file_path = "/kaggle/working/ppe_data.yaml"
    with open(yaml_file_path, "w") as f:
        f.write(yaml_content.strip())

    print(f"Configuration written to {yaml_file_path}:\n")
    print(yaml_content)

Configuration written to /kaggle/working/ppe_data.yaml:


train: /kaggle/input/datasets/ndomalau/personal-protective-equipment-ppe-dataset/Personal-Protective-Equipment (PPE) Dataset/train/images
val: /kaggle/input/datasets/ndomalau/personal-protective-equipment-ppe-dataset/Personal-Protective-Equipment (PPE) Dataset/valid/images

nc: 5
names: ['helmet', 'no_helmet', 'no_vest', 'person', 'vest']



# Train YOLOv8

In [11]:
results = model.train(
    data="/kaggle/working/ppe_data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    hsv_h=0.015,       # Adjusts hue to generalize across color variations
    hsv_s=0.7,         # Adjusts saturation to simulate different environmental conditions
    hsv_v=0.4,         # Adjusts brightness to handle harsh outdoor exposure levels
    degrees=10.0,      # Random rotation up to 10 degrees
    translate=0.1,     # Shifts the image horizontally and vertically
    scale=0.5,         # Randomly scales to simulate objects at different distances
    shear=2.0,         # Adds perspective-like distortions (mimics camera angles)
    fliplr=0.5,        # 50% probability to flip images horizontally
    mosaic=1.0,        # Combines 4 images into one, heavily improving small-object detection
    erasing=0.4,       # Randomly erases patches to train the model to handle occlusion
    project="ppe_yolov8",
    name="yolov8_ppe",
    pretrained=True,
    patience=10,
    save=True,
    plots=True
)

Ultralytics 8.4.153 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/ppe_data.yaml, degrees=10.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/runs/detect/ppe_yolov8/yolov8_ppe-3/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale

Load the Best Model

In [13]:
best_model = YOLO("/kaggle/working/runs/detect/ppe_yolov8/yolov8_ppe-4/weights/best.pt")

print("Best YOLOv8 model loaded.")

Best YOLOv8 model loaded.


Validation

In [14]:
metrics = best_model.val(
    data="/kaggle/working/ppe_data.yaml",
    imgsz=640,
    batch=16,
    device=0,
    plots=True
)

Ultralytics 8.4.153 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 72 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 108.6±40.4 MB/s, size: 53.4 KB)
val: Scanning /kaggle/input/datasets/ndomalau/personal-protective-equipment-ppe-dataset/Personal-Protective-Equipment (PPE) Dataset/valid/labels... 406 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 406/406 1.1Kit/s 0.4s<0.1s
WARNING ⚠️ val: Cache directory /kaggle/input/datasets/ndomalau/personal-protective-equipment-ppe-dataset/Personal-Protective-Equipment (PPE) Dataset/valid is not writable, cache not saved.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 4.5it/s 5.8s0.1s
                   all        406       3095      0.904      0.852      0.906      0.577
                helmet        246        531      0.935      0.853      0.903      0.576
             no_helmet  

Extract Main Metrics

In [15]:
print("Detection Metrics")
print("=" * 50)

print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")

Detection Metrics
mAP50:     0.9055
mAP50-95:  0.5772


In [16]:
print(metrics.results_dict)

{'metrics/precision(B)': 0.9043180478294317, 'metrics/recall(B)': 0.8519771317711096, 'metrics/mAP50(B)': 0.9055215737943602, 'metrics/mAP50-95(B)': 0.5771963527247201, 'fitness': 0.5771963527247201}


Per-Class Performance

In [22]:
    print("\n[Per-Class Performance Breakdown]")
    class_names = [model.names[i] for i in range(len(model.names))]
    print(f"{'Class Name':<15} | {'P (%)':<8} | {'R (%)':<8} | {'mAP@0.5 (%)':<12} | {'mAP@0.5:0.95 (%)':<16}")
    print("-" * 70)
    
    for idx, c_name in enumerate(class_names):
        p = metrics.box.p[idx] * 100 if len(metrics.box.p) > idx else 0.0
        r = metrics.box.r[idx] * 100 if len(metrics.box.r) > idx else 0.0
        m50 = metrics.box.all_ap[idx, 0] * 100 if metrics.box.all_ap.shape[1] > 0 else 0.0
        m95 = metrics.box.ap[idx] * 100 if len(metrics.box.ap) > idx else 0.0
        print(f"{c_name:<15} | {p:<8.2f} | {r:<8.2f} | {m50:<12.2f} | {m95:<16.2f}")



[Per-Class Performance Breakdown]
Class Name      | P (%)    | R (%)    | mAP@0.5 (%)  | mAP@0.5:0.95 (%)
----------------------------------------------------------------------
helmet          | 93.54    | 85.31    | 90.28        | 57.55           
no_helmet       | 88.08    | 78.78    | 85.72        | 49.42           
no_vest         | 86.79    | 84.00    | 88.52        | 50.39           
person          | 90.92    | 89.06    | 93.62        | 64.15           
vest            | 92.83    | 88.85    | 94.62        | 67.09           


Precision and Recall

In [ ]:
print("Validation Results")
print("=" * 50)

for key, value in metrics.results_dict.items():
    print(f"{key}: {value}")

Training Curves

In [ ]:
plots=True

In [ ]:
from IPython.display import Image, display

display(
    Image(
        filename="ppe_yolov8/yolov8_ppe/results.png"
    )
)

Confusion Matrix

In [ ]:
display(
    Image(
        filename="ppe_yolov8/yolov8_ppe/confusion_matrix_normalized.png"
    )
)

Run Detection on Test Images

In [ ]:
TEST_SOURCE = SPLIT_DIRS["test"]["images"]

print("Test images:", TEST_SOURCE)

YOLOv8 Test Prediction

In [ ]:
test_results = best_model.predict(
    source=TEST_SOURCE,
    imgsz=640,
    conf=0.25,
    iou=0.50,
    save=True,
    save_txt=True,
    save_conf=True,
    project="ppe_predictions",
    name="test_results"
)

Visualize YOLO Predictions

In [ ]:
result = test_results[0]

annotated_image = result.plot()

plt.figure(figsize=(12, 8))
plt.imshow(cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("YOLOv8 PPE Detection")
plt.show()